# ![icon](./images/uva-icon-57x57.png) WEEK 08 Snowflake

| Working Efficiently With Software |
| :--: |
| Data Engineering |
| School of Data Science |
| University of Virginia |

# ![icon](./images/uva-icon-57x57.png) Announcements & Agenda

---

**NB!: Please remember your PRs will 'BOUNCE'**
* **If the description is not filled out properly.**
* **If your branch name is not named properly.**

---

No lab assignment this week.  However, we're past mid point so I am going to push out the next labs asynchronously to avoid a crunch at the end.

## T-Shirt sizing the remaining labs.
- Setting up snowflake connector.  **M/L**
- Dockerizing our pipeline.  **XL** **<===== AWS ⬆️ == Snowflake ⬇️ ===**
- Git repo in snowflake.  **M**
- DBT-expectations **M**
- Snowflake CORTEX.  **S**

# ![icon](./images/uva-icon-57x57.png) Logging into Snowflake

## First time logging in
* Go to this link:
```
https://app.snowflake.com/adilksi/fqa59308/
```

* You will have to set up 2fa

# ![icon](./images/uva-icon-57x57.png) Check you can run SQL

## Ensure you are using the right role.
* Navigate to Project/Workspaces
* Create a new **sql** worksheet
* Select Databases on the left view

## Select your ROLE and Warehouse
* Over your new sql file look for the icon just left of the little house
* Select `DS5111_STUDENT_ROLE`
* Select warehouse `DS5111_WH`


## Set Database and Schema
The DB and Schema should have been set for you
```sql
USE DATABASE DS5111_DB;
USE SCHEMA TXT1SR;
```

## Create a new table for our test
```sql
CREATE OR REPLACE TEMPORARY TABLE json_playground (
    raw_data VARIANT
);
```

## Insert json type of data into an VARIANT field
```sql
INSERT INTO json_playground (raw_data)
SELECT PARSE_JSON('{"video_id": "uva_demo_123", "metrics": {"views": 1500, "likes": 42}, "tags": ["edu", "data-eng"]}');
```

## Inspect the data, NOTE how we didn't have to define any columns
```sql
SELECT 
    raw_data:video_id::STRING       AS video_id,
    raw_data:metrics.views::INT     AS view_count,
    raw_data:tags[0]::STRING        AS primary_tag
FROM json_playground;
```



# ![icon](./images/uva-icon-57x57.png)  Snowflake

**Separation of Compute and Storage**

**Problem:** If a data scientist runs a massive machine learning query that consumes 100% of the CPU, your production ingestion pipeline freezes.

**Solution:** Snowflake solved this by completely **decoupling Storage from Compute**.

```mermaid
graph TD
    %% Traditional Architecture Section
    subgraph Legacy ["TRADITIONAL COUPLED SCALING (Shared-Nothing Architecture)"]
        direction LR
        subgraph Node1 ["Node 1"]
            CPU1(CPU) <--> DB1[(DB Storage)]
        end
        subgraph Node2 ["Node 2"]
            CPU2(CPU) <--> DB2[(DB Storage)]
        end
        subgraph Node3 ["Node 3"]
            CPU3(CPU) <--> DB3[(DB Storage)]
        end
        subgraph Node4 ["Node 4"]
            CPU4(CPU) <--> DB4[(DB Storage)]
        end
        subgraph Node5 ["Node 5"]
            CPU5(CPU) <--> DB5[(DB Storage)]
        end
        Node1 <-->|Complex Network Re-Sharding| Node5
    end

    %% Snowflake Architecture Section
    subgraph Modern ["SNOWFLAKE DECOUPLED SCALING (Multi-Cluster Shared Data)"]
        direction TB
        
        subgraph ComputePool ["COMPUTE TIER: 10 Stateless Elastic CPU Nodes"]
            direction LR
            C1(CPU 1)  --- C2(CPU 2)  --- C3(CPU 3)  --- C4(CPU 4)  --- C5(CPU 5)
            C6(CPU 6)  --- C7(CPU 7)  --- C8(CPU 8)  --- C9(CPU 9)  --- C10(CPU 10)
        end

        subgraph StoragePool ["STORAGE TIER: 5 Centralized Shared DB Storage Volumes"]
            direction LR
            S1[(DB Vol 1)] --- S2[(DB Vol 2)] --- S3[(DB Vol 3)] --- S4[(DB Vol 4)] --- S5[(DB Vol 5)]
        end

        ComputePool ==>|"Independent Elastic Reads"| StoragePool
    end
```

# ![icon](./images/uva-icon-57x57.png)  Warehouses are independent

**UI:  [Compute/Warehouses](https://app.snowflake.com/adilksi/fqa59308/#/compute/warehouses?whtype=all&status=all&size=all&columns=name%2Cstatus%2Csize%2Ctype%2Cclusters%2Crunning%2Cqueued%2Ctables%2Cowner%2ClastResumedTime%2CresourceConstraint)
* You can have multiple warehouses in different sizes, (T-shirt sized)
* Auto-Suspend Rule - typical to be set for **5 to 10** minutes.  Stops machines if not used.

# ![icon](./images/uva-icon-57x57.png)  Security and ROLES


## LAYERS
* **Identity:** Users
* **Logical:** Roles
* **Resource:** Compute and Storage

**Golden Rule:** All custom functional roles must roll up to **SYSADMIN**

```mermaid
graph TD
    %% System Administration Core
    SYSADMIN["SYSADMIN (System Admin Role)"]

    %% Identity Layer (Users)
    subgraph Identity ["Identity Layer: Individual Users"]
        U_Sarah(["User: Sarah (Data Engineer)"])
        U_Alex(["User: Alex (Data Scientist)"])
    end

    %% Logical Layer (Functional Job Roles)
    subgraph Roles ["Logical Layer: Functional Job Roles"]
        DE_ROLE["DATA_ENGINEER_ROLE"]
        DS_ROLE["DATA_SCIENTIST_ROLE"]
    end

    %% Identity to Role Mappings
    U_Sarah --> DE_ROLE
    U_Alex --> DS_ROLE

    %% The Enforced Operational Hierarchy Roll-up
    DE_ROLE --> SYSADMIN
    DS_ROLE --> SYSADMIN

    %% Resource Layer: Compute Compute
    subgraph Compute ["Resource Layer: Compute (Virtual Warehouses)"]
        DE_WH["DE_LOAD_WH (X-Large Ingestion Cluster)"]
        DS_WH["DS_ANALYTICS_WH (Medium Variable Cluster)"]
    end

    %% Resource Layer: Storage
    subgraph Storage ["Resource Layer: Storage (Database & Schemas)"]
        subgraph DB ["DS5111_DATA_WAREHOUSE"]
            RAW["RAW_LANDING (Schema)"]
            PROD["ANALYTICS_PROD (Schema)"]
            SANDBOX["DATA_SCIENCE_SANDBOX (Schema)"]
        end
    end

    %% Data Engineer Access Vectors
    DE_ROLE ==>|"USAGE, OPERATE (Power to run heavy loads)"| DE_WH
    DE_ROLE ==>|"ALL PRIVILEGES (Read/Write Pipelines)"| RAW
    DE_ROLE ==>|"ALL PRIVILEGES (DDL Transform)"| PROD

    %% Data Scientist Access Vectors
    DS_ROLE ==>|"USAGE (Power to run model training)"| DS_WH
    DS_ROLE ==>|"SELECT (Read-Only Consumer access)"| PROD
    DS_ROLE ==>|"ALL PRIVILEGES (Full Sandbox Freedom)"| SANDBOX

    %% Styling Elements
    classDef user fill:#e1f5fe,stroke:#0288d1,stroke-width:2px;
    classDef role fill:#ede7f6,stroke:#5e35b1,stroke-width:2px;
    classDef sys fill:#fff3e0,stroke:#f57c00,stroke-width:2px;
    
    class U_Sarah,U_Alex user;
    class DE_ROLE,DS_ROLE role;
    class SYSADMIN sys;
```

## Separation of concerns
### If **Alex runs a query that bogs down compute** on his warehouse, **Sarah isn't impacted** because she uses a different warehouse.

# ![icon](./images/uva-icon-57x57.png) Least Privilege

Read write access is controlled via the ROLES.

## All users in a role have the same access. So for example:
* Data Eng Role may have write/drop access to some tables
* Date Sci Role may have read only access to those same tables.

When you have a new Data Scientist, you add them to a role, and all the permissions are inherited.

# ![icon](./images/uva-icon-57x57.png) Users acting in Roles grant access to peers by default

If a Data Scientist using the `DATA_SCIENTIST_ROLE` creates a new table, all other data scientists have access as well